# CareLoop Clinics: Part 4, GenAI call-centre notes

Upload `followup_notes.csv` and `appointments_clean.csv` before running Task 4.2 and 4.4.
Run each cell **once, in order** (re-running 4.1-4.3 inflates the API call counter).
The API key is typed at run time and is never saved in this notebook.

## Task 4.1: One validated call

In [ ]:
!pip install -q google-genai pydantic
import json, time
import pandas as pd
from typing import Literal
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError
from google.colab import files

uploaded = files.upload()   # followup_notes.csv and appointments_clean.csv

In [ ]:
ALLOWED = ["forgot_or_no_reminder", "distance_or_transport", "cost_or_fee",
           "illness_or_emergency", "work_or_schedule", "other"]

SCHEMA = {   # root_cause is a plain string on purpose (NOT an enum); Pydantic enforces the list later
    "type": "object",
    "properties": {"root_cause": {"type": "string"}, "confidence": {"type": "number"},
                   "summary": {"type": "string"}, "recommended_action": {"type": "string"}},
    "required": ["root_cause", "confidence", "summary", "recommended_action"]}

# Prompt built with the CRAFT parts
SYSTEM_TEXT = f"""Context: You read short call-centre notes about patients of an Indian clinic chain who missed an appointment.
Role: You are a careful clinic operations analyst.
Action: Decide the single main reason the patient missed the appointment and suggest one follow-up action.
Rules: root_cause MUST be exactly one of {ALLOWED}. If the note is unclear or fits none of them, answer "other" rather than guessing.
confidence is a number from 0 to 1. summary has at most 25 words.
Tone: Neutral, factual and brief.
Format: Return only JSON with the keys root_cause, confidence, summary, recommended_action."""

API_KEY = input("Paste your key: ")
client = genai.Client(api_key=API_KEY)
MODEL_NAME = "PUT-THE-FLASH-MODEL-NAME-FROM-SESSION-22.1-HERE"

api_calls = 0      # counts every API call (needed in Task 4.3c)
last_raw = None    # exact text of the most recent reply (needed for raw_response)

def analyse_note(note_text, system_text=SYSTEM_TEXT):
    """Send one note to Gemini and return the parsed dictionary."""
    global api_calls, last_raw
    api_calls += 1
    last_raw = None
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=note_text,
        config=types.GenerateContentConfig(
            system_instruction=system_text, temperature=0, max_output_tokens=1000,
            response_mime_type="application/json", response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(thinking_level="minimal")))
    last_raw = response.text          # keep the reply exactly as returned
    return json.loads(response.text)

In [ ]:
notes = pd.read_csv("followup_notes.csv")
print(notes.columns.tolist())
note_n001 = notes.loc[notes["note_id"] == "N001", "note_text"].iloc[0]
result = analyse_note(note_n001)
print(result)    # root_cause should be forgot_or_no_reminder

**What this shows:** (fill in after running: the four fields returned for N001 and its root_cause.)

## Task 4.2: Validate and process the first 10 notes

In [ ]:
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def check_insight(data):
    """Pydantic check + allowed-list check. Raises ValidationError / ValueError if bad."""
    insight = FollowupInsight(**data)
    if insight.root_cause not in ALLOWED:
        raise ValueError(f"root_cause '{insight.root_cause}' is not in the allowed list")
    return insight

FIELDS = ["root_cause", "confidence", "summary", "recommended_action"]
first10 = pd.read_csv("followup_notes.csv").head(10)

records = []
for _, row in first10.iterrows():
    rec = {"note_id": row["note_id"], "appointment_id": row["appointment_id"],
           "root_cause": "", "confidence": "", "summary": "", "recommended_action": "",
           "validation_status": "", "raw_response": ""}
    note_text = row["note_text"]

    # (b) input validation FIRST: isna, then strip. Blank notes never reach the API.
    if pd.isna(note_text) or str(note_text).strip() == "":
        rec.update(root_cause="other", confidence=0.0, summary="No note recorded",
                   recommended_action="Re-call patient", validation_status="skipped_blank",
                   raw_response="")
    else:
        try:
            data = analyse_note(str(note_text))
            rec["raw_response"] = last_raw
            if isinstance(data, dict):
                for k in FIELDS:                     # copy whatever fields the reply has
                    if k in data and data[k] is not None:
                        rec[k] = data[k]
            check_insight(data)
            rec["validation_status"] = "valid"
        except Exception as e:                       # one bad reply never stops the loop
            rec["validation_status"] = "invalid"
            if last_raw is not None:
                rec["raw_response"] = last_raw
            print(f"{row['note_id']}: invalid ({type(e).__name__})")
        time.sleep(13)                               # (c) free-tier rate limit

    records.append(rec)

notes_analysed = pd.DataFrame(records, columns=["note_id", "appointment_id", "root_cause", "confidence",
                                                "summary", "recommended_action",
                                                "validation_status", "raw_response"])
notes_analysed.to_csv("notes_analysed.csv", index=False)   # (d)

print(notes_analysed.shape)                                # expect (10, 8)
print(notes_analysed["validation_status"].value_counts())
print("API calls so far:", api_calls)                      # expect 10 (1 from 4.1 + 9 here)
print(notes_analysed[["note_id", "root_cause", "confidence", "validation_status"]].to_string(index=False))
files.download("notes_analysed.csv")

**What this shows:** (fill in after running) how many rows are `valid`, `invalid` and `skipped_blank`; that N004 was skipped with no API call;
and how many of the 9 non-blank notes match the expected root causes in the brief (at least 7 of 9 needed).
Invalid rows are kept exactly as the model returned them, not repaired.

## Task 4.3: Break it on purpose

In [ ]:
# (a) two deliberately bad replies, each caught without crashing
bad_confidence = {"root_cause": "other", "confidence": 1.7,
                  "summary": "Patient was unwell", "recommended_action": "Call back next week"}
bad_missing = {"root_cause": "other", "confidence": 0.5, "summary": "Patient was unwell"}   # no recommended_action

for name, bad in [("confidence = 1.7", bad_confidence), ("missing recommended_action", bad_missing)]:
    try:
        FollowupInsight(**bad)
        print(name, "-> unexpectedly passed")
    except ValidationError as e:
        print(f"--- {name} ---")
        print(e, "\n")

In [ ]:
# (b) V1 (bare prompt) vs V2 (CRAFT prompt with allowed values), same schema and settings
V1_TEXT = "Classify this note"

def get_text(nid):
    return notes.loc[notes["note_id"] == nid, "note_text"].iloc[0]

rows = []
for nid in ["N007", "N009"]:
    for label, sys_text in [("V1", V1_TEXT), ("V2", SYSTEM_TEXT)]:
        try:
            d = analyse_note(get_text(nid), system_text=sys_text)
            rc = d.get("root_cause")
        except Exception as e:
            rc = f"ERROR: {type(e).__name__}"
        rows.append({"note": nid, "prompt": label, "root_cause_returned": rc,
                     "in_allowed_list": rc in ALLOWED})
        time.sleep(13)

compare = pd.DataFrame(rows)
print(compare.to_string(index=False))

In [ ]:
# (c) total API calls across Tasks 4.1-4.3
print("Total API calls used in Tasks 4.1-4.3:", api_calls)   # expect 14 (12 if you reuse 4.2 for V2), below 25

**What this shows:** (fill in after running)
- (a) The `confidence = 1.7` reply fails with "Input should be less than or equal to 1"; the reply with no `recommended_action` fails with "Field required". Both are caught by `try/except`.
- (b) Table above. State which prompt you would deploy. Normally V2, because it lists the allowed values and tells the model to answer `other` when unsure, so its replies stay inside the allowed list, while V1 can return any free-text label.
- (c) Total API calls: (write the number printed above).

## Task 4.4: Recommendation (merge and cross-tab)

In [ ]:
na = pd.read_csv("notes_analysed.csv")
ap = pd.read_csv("appointments_clean.csv")

merged = na.merge(ap, on="appointment_id", how="inner")
print("Merged rows:", len(merged))                      # expect 10

print(pd.crosstab(merged["root_cause"], merged["reminder_sent"]))
# expect: both forgot_or_no_reminder notes (N001, N007) under reminder_sent = No; 5 of the 10 notes have reminder_sent = No